# Chapter 10: Creating Text Embedding Models - Medium Tasks

This notebook covers advanced embedding training: implementing training loops, MTEB evaluation, negative sampling strategies, and hyperparameter optimization.

## Setup

Run all cells in this section to set up the environment and load the model.

Before running these cells, review the concepts from the main Chapter 10 notebook (00_Start_Here.ipynb).

### [Optional] - Installing Packages on Google Colab

If you are viewing this notebook on Google Colab, uncomment and run the following code to install dependencies.

**Note**: Use a GPU for this notebook. In Google Colab, go to Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4.

In [ ]:
# %%capture
# !pip install -q accelerate>=0.27.2 transformers>=4.38.2
# !pip install -q sentence-transformers>=3.0.0 mteb>=1.1.2 datasets>=2.18.0

### Model Loading

In [ ]:
import torch
from datasets import load_dataset, Dataset
from sentence_transformers import SentenceTransformer, losses
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

### Helper Functions

In [ ]:
def create_evaluator():
    """Create standard evaluator for consistent evaluation"""
    val_sts = load_dataset('glue', 'stsb', split='validation')
    return EmbeddingSimilarityEvaluator(
        sentences1=val_sts["sentence1"],
        sentences2=val_sts["sentence2"],
        scores=[score/5 for score in val_sts["label"]],
        main_similarity="cosine"
    )

## Challenges

Complete the following tasks.

### Level: Medium

**About This Task:**

Training embeddings requires choosing the right loss function for your task. Different losses optimize for different objectives.

#### Medium Task 1: Comparing Loss Functions in Training

### Instructions

1. Prepare data for different loss functions
2. Train models with CosineSimilarityLoss and MultipleNegativesRankingLoss
3. Track training metrics
4. Compare final evaluation scores
5. Analyze which loss function works better for semantic similarity

Load and prepare training data.

In [ ]:
# Load MNLI dataset
train_data = load_dataset("glue", "mnli", split="train").select(range(10_000))
train_data = train_data.remove_columns("idx")

### Prepare Data for CosineSimilarityLoss

In [ ]:
# Transform labels: entailment=1.0, neutral/contradiction=0.0
mapping = {0: 1.0, 1: 0.0, 2: 0.0}

cosine_dataset = Dataset.from_dict({
    "sentence1": train_data["premise"],
    "sentence2": train_data["hypothesis"],
    "label": [mapping[label] for label in train_data["label"]]
})

print(f"CosineSimilarityLoss dataset: {len(cosine_dataset)} examples")
print(f"Example: {cosine_dataset[0]}")

### Prepare Data for MultipleNegativesRankingLoss

Your task: Prepare data with anchor, positive, and negative.

In [ ]:
import random

# Filter entailment pairs only
entailment_data = train_data.filter(lambda x: x['label'] == 0)

# Fill in: Create soft negatives by shuffling
negatives = list(entailment_data["hypothesis"])
random.shuffle(negatives)

mnr_dataset = Dataset.from_dict({
    "anchor": entailment_data["premise"],
    "positive": entailment_data["hypothesis"],
    "negative": negatives
})

print(f"\nMNR dataset: {len(mnr_dataset)} examples")
print(f"Example: {mnr_dataset[0]}")

### Train with CosineSimilarityLoss

In [ ]:
# Create model
model_cosine = SentenceTransformer('bert-base-uncased')

# Loss function
loss_cosine = losses.CosineSimilarityLoss(model=model_cosine)

# Training arguments
args_cosine = SentenceTransformerTrainingArguments(
    output_dir="cosine_model",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    warmup_steps=50,
    fp16=True,
    logging_steps=50,
)

# Create evaluator
evaluator = create_evaluator()

# Train
trainer_cosine = SentenceTransformerTrainer(
    model=model_cosine,
    args=args_cosine,
    train_dataset=cosine_dataset,
    loss=loss_cosine,
    evaluator=evaluator
)

trainer_cosine.train()

In [ ]:
# Evaluate CosineSimilarityLoss model
results_cosine = evaluator(model_cosine)
print("\nCosineSimilarityLoss Results:")
print(f"Spearman Cosine: {results_cosine['spearman_cosine']:.4f}")

Clear GPU memory before training next model.

In [ ]:
import gc

del trainer_cosine, model_cosine, loss_cosine
gc.collect()
torch.cuda.empty_cache()

### Train with MultipleNegativesRankingLoss

Your task: Complete the training configuration.

In [ ]:
# Create model
model_mnr = SentenceTransformer('bert-base-uncased')

# Fill in: Create MNR loss
loss_mnr = losses.MultipleNegativesRankingLoss(model=model_mnr)

# Fill in: Create training arguments (same as cosine)
args_mnr = SentenceTransformerTrainingArguments(
    output_dir="mnr_model",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    warmup_steps=50,
    fp16=True,
    logging_steps=50,
)

# Train
trainer_mnr = SentenceTransformerTrainer(
    model=model_mnr,
    args=args_mnr,
    train_dataset=mnr_dataset,
    loss=loss_mnr,
    evaluator=evaluator
)

trainer_mnr.train()

In [ ]:
# Evaluate MNR model
results_mnr = evaluator(model_mnr)
print("\nMultipleNegativesRankingLoss Results:")
print(f"Spearman Cosine: {results_mnr['spearman_cosine']:.4f}")

### Compare Results

In [ ]:
print("\n" + "="*60)
print("Comparison:")
print(f"CosineSimilarityLoss:        {results_cosine['spearman_cosine']:.4f}")
print(f"MultipleNegativesRankingLoss: {results_mnr['spearman_cosine']:.4f}")
print(f"Difference: {abs(results_cosine['spearman_cosine'] - results_mnr['spearman_cosine']):.4f}")

### Questions

1. Which loss function performed better? Why might that be?

2. MNR uses in-batch negatives. How does batch size affect MNR performance?

3. When would you choose SoftmaxLoss over these two loss functions?

**About This Task:**

MTEB (Massive Text Embedding Benchmark) provides standardized evaluation across many tasks. Understanding how to use it is crucial for comparing models.

#### Medium Task 2: MTEB Benchmark Evaluation

### Instructions

1. Run MTEB evaluation on a classification task
2. Compare trained vs base model performance
3. Analyze accuracy and F1 scores
4. Test on multiple MTEB tasks
5. Understand which tasks benefit most from training

In [ ]:
from mteb import MTEB

Evaluate base model on Banking77 classification.

In [ ]:
# Load base model
base_model = SentenceTransformer('bert-base-uncased')

# Create evaluation task
evaluation = MTEB(tasks=["Banking77Classification"])

# Run evaluation
base_results = evaluation.run(base_model)

print("\nBase Model Results:")
print(f"Accuracy: {base_results[0]['scores']['test'][0]['main_score']:.4f}")

Evaluate your trained model.

In [ ]:
# Use the MNR model from Task 1
trained_results = evaluation.run(model_mnr)

print("\nTrained Model Results:")
print(f"Accuracy: {trained_results[0]['scores']['test'][0]['main_score']:.4f}")

### Task 2a: Evaluate on Multiple Tasks

Your task: Run evaluation on 2-3 different MTEB tasks and compare.

In [ ]:
# Fill in: Choose 2-3 MTEB tasks
# Options: "ToxicConversationsClassification", "EmotionClassification", etc.

multi_eval = MTEB(tasks=[
    "Banking77Classification",
    # Add more tasks here
])

# Note: This may take several minutes
# multi_results = multi_eval.run(model_mnr)

### Task 2b: Analyze Results Structure

Explore what information MTEB returns.

In [ ]:
# Examine result structure
print("Result keys:")
for key in trained_results[0].keys():
    print(f"  {key}")

print("\nDetailed metrics:")
test_scores = trained_results[0]['scores']['test'][0]
for metric, value in test_scores.items():
    if isinstance(value, (int, float)):
        print(f"  {metric}: {value:.4f}")

### Questions

1. Did training improve MTEB scores? By how much?

2. Why might a model trained on MNLI transfer well to Banking77?

3. What's the difference between accuracy and F1 in the results? When does F1 matter more?

**About This Task:**

Negative samples are crucial for contrastive learning. Different strategies for creating negatives affect model quality.

#### Medium Task 3: Negative Sampling Strategies

### Instructions

1. Implement random negative sampling
2. Implement hard negative sampling (similar but wrong)
3. Compare training with different negative strategies
4. Measure the impact on evaluation metrics
5. Analyze which strategy produces better embeddings

In [ ]:
# Load fresh data
train_data = load_dataset("glue", "mnli", split="train").select(range(5_000))
entailment_data = train_data.filter(lambda x: x['label'] == 0)

print(f"Entailment pairs: {len(entailment_data)}")

### Strategy 1: Random Negatives

In [ ]:
import random

def create_random_negatives(data):
    """Create negatives by random shuffling"""
    negatives = list(data["hypothesis"])
    random.shuffle(negatives)
    
    return Dataset.from_dict({
        "anchor": data["premise"],
        "positive": data["hypothesis"],
        "negative": negatives
    })

random_neg_dataset = create_random_negatives(entailment_data)
print("Random negatives example:")
print(random_neg_dataset[0])

### Strategy 2: Hard Negatives

Your task: Create hard negatives using contradiction pairs (label=2).

In [ ]:
def create_hard_negatives(entailment_data, full_data):
    """Create hard negatives from contradiction pairs"""
    # Fill in: Filter contradiction pairs
    contradiction_data = full_data.filter(lambda x: x['label'] == 2)
    
    # Match each entailment with a contradiction
    # For simplicity, we'll pair them sequentially
    min_len = min(len(entailment_data), len(contradiction_data))
    
    return Dataset.from_dict({
        "anchor": entailment_data["premise"][:min_len],
        "positive": entailment_data["hypothesis"][:min_len],
        "negative": contradiction_data["hypothesis"][:min_len]
    })

hard_neg_dataset = create_hard_negatives(entailment_data, train_data)
print("\nHard negatives example:")
print(hard_neg_dataset[0])

### Compare Negative Quality

Use a pre-trained model to measure how different negatives are.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Load pre-trained model for comparison
comparison_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def measure_negative_quality(dataset, num_samples=50):
    """Measure similarity between anchor and negatives"""
    similarities = []
    
    for i in range(min(num_samples, len(dataset))):
        anchor_emb = comparison_model.encode([dataset[i]["anchor"]])
        negative_emb = comparison_model.encode([dataset[i]["negative"]])
        
        sim = cosine_similarity(anchor_emb, negative_emb)[0][0]
        similarities.append(sim)
    
    return np.mean(similarities), np.std(similarities)

random_mean, random_std = measure_negative_quality(random_neg_dataset)
hard_mean, hard_std = measure_negative_quality(hard_neg_dataset)

print("\nNegative Quality (lower similarity = better negative):")
print(f"Random negatives: {random_mean:.4f} ± {random_std:.4f}")
print(f"Hard negatives:   {hard_mean:.4f} ± {hard_std:.4f}")

### Task 3a: Train with Different Negatives

Your task: Train a small model with each strategy and compare.

In [ ]:
# This is a template - training takes time, so you may want to reduce data size

def train_with_negatives(dataset, output_dir):
    """Train model with given negative strategy"""
    model = SentenceTransformer('bert-base-uncased')
    loss = losses.MultipleNegativesRankingLoss(model=model)
    
    args = SentenceTransformerTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=1,
        per_device_train_batch_size=16,
        warmup_steps=20,
        fp16=True,
        logging_steps=50,
    )
    
    evaluator = create_evaluator()
    
    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=dataset,
        loss=loss,
        evaluator=evaluator
    )
    
    trainer.train()
    results = evaluator(model)
    
    return model, results

# Uncomment to train:
# model_random, results_random = train_with_negatives(random_neg_dataset, "random_neg")
# model_hard, results_hard = train_with_negatives(hard_neg_dataset, "hard_neg")

### Questions

1. Which negative strategy produced lower anchor-negative similarity?

2. Why might hard negatives (contradictions) be better than random negatives?

3. What would be an even harder negative? How would you create "semi-hard" negatives?

**About This Task:**

Hyperparameters like batch size, learning rate, and warmup steps significantly affect training. Systematic testing finds optimal settings.

#### Medium Task 4: Hyperparameter Comparison

### Instructions

1. Test different batch sizes
2. Test different numbers of warmup steps
3. Test different numbers of epochs
4. Track evaluation scores for each configuration
5. Identify which hyperparameters have the biggest impact

In [ ]:
# Prepare small dataset for quick experiments
small_train = load_dataset("glue", "mnli", split="train").select(range(2_000))
small_train = small_train.filter(lambda x: x['label'] == 0)

small_dataset = Dataset.from_dict({
    "anchor": small_train["premise"],
    "positive": small_train["hypothesis"],
})

print(f"Small dataset: {len(small_dataset)} examples")

Create a function to test hyperparameters.

In [ ]:
def test_hyperparameters(batch_size, warmup_steps, num_epochs, name):
    """Train with specific hyperparameters and evaluate"""
    model = SentenceTransformer('bert-base-uncased')
    loss = losses.MultipleNegativesRankingLoss(model=model)
    
    args = SentenceTransformerTrainingArguments(
        output_dir=f"hyperparam_{name}",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        warmup_steps=warmup_steps,
        fp16=True,
        logging_steps=20,
    )
    
    evaluator = create_evaluator()
    
    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=small_dataset,
        loss=loss,
        evaluator=evaluator
    )
    
    trainer.train()
    results = evaluator(model)
    
    # Clean up
    del model, trainer
    torch.cuda.empty_cache()
    
    return results['spearman_cosine']

### Test Different Batch Sizes

In [ ]:
# Test batch sizes: 8, 16, 32
batch_results = {}

for bs in [8, 16, 32]:
    print(f"\nTesting batch_size={bs}...")
    score = test_hyperparameters(
        batch_size=bs,
        warmup_steps=20,
        num_epochs=1,
        name=f"bs{bs}"
    )
    batch_results[bs] = score
    print(f"Batch size {bs}: {score:.4f}")

print("\n" + "="*60)
print("Batch Size Comparison:")
for bs, score in batch_results.items():
    print(f"  {bs:2d}: {score:.4f}")

### Task 4a: Test Warmup Steps

Your task: Test different warmup step values.

In [ ]:
# Fill in: Test warmup_steps values: 0, 20, 50
warmup_results = {}

for warmup in [0, 20, 50]:
    print(f"\nTesting warmup_steps={warmup}...")
    score = test_hyperparameters(
        batch_size=16,
        warmup_steps=warmup,
        num_epochs=1,
        name=f"warmup{warmup}"
    )
    warmup_results[warmup] = score
    print(f"Warmup {warmup}: {score:.4f}")

print("\n" + "="*60)
print("Warmup Steps Comparison:")
for warmup, score in warmup_results.items():
    print(f"  {warmup:2d}: {score:.4f}")

### Visualize Results

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Batch size plot
ax1.plot(list(batch_results.keys()), list(batch_results.values()), marker='o')
ax1.set_xlabel('Batch Size')
ax1.set_ylabel('Spearman Correlation')
ax1.set_title('Effect of Batch Size')
ax1.grid(True)

# Warmup plot
ax2.plot(list(warmup_results.keys()), list(warmup_results.values()), marker='o', color='orange')
ax2.set_xlabel('Warmup Steps')
ax2.set_ylabel('Spearman Correlation')
ax2.set_title('Effect of Warmup Steps')
ax2.grid(True)

plt.tight_layout()
plt.show()

### Questions

1. Which batch size worked best? Why might larger batches help MultipleNegativesRankingLoss?

2. Did warmup steps improve performance? What does warmup prevent?

3. If you had to choose between training longer (more epochs) or larger batches, which would you pick for embedding models?